<a href="https://colab.research.google.com/github/Rainbow-CC/my-ai-present-test/blob/master/model_train_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 安装 unsloth

In [ ]:
# 安装 unsloth 包。unsloth 是一个用于微调大型语言模型（LLM）的工具，可以让模型运行更快、占用更少内存。
!pip install unsloth

# 卸载当前已安装的 unsloth 包（如果已安装），然后从 GitHub 的源代码安装最新版本。
# 这样可以确保我们使用的是最新功能和修复。
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

# 安装 bitsandbytes 和 unsloth_zoo 包。
# bitsandbytes 是一个用于量化和优化模型的库，可以帮助减少模型占用的内存。
# unsloth_zoo 可能包含了一些预训练模型或其他工具，方便我们使用。
!pip install bitsandbytes unsloth_zoo


In [ ]:
!pip uninstall unsloth unsloth_zoo bitsandbytes -y
# 强制安装 unsloth 及其所有配套依赖，不需要重复运行。
!pip install --no-cache-dir --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir --upgrade "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install --no-cache-dir bitsandbytes

## 加载模型

In [ ]:
from unsloth import FastLanguageModel  # 导入FastLanguageModel类，用来加载和使用模型
import torch  # 导入torch工具，用于处理模型的数学运算

max_seq_length = 1024  # 设置模型处理文本的最大长度，相当于给模型设置一个“最大容量”
dtype = None  # 设置数据类型，让模型自动选择最适合的精度
load_in_4bit = True  # 使用4位量化来节省内存，就像把大箱子压缩成小箱子

# 加载预训练模型，并获取tokenizer工具
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Llama-8B",  # 指定要加载的模型名称
    max_seq_length=max_seq_length,  # 使用前面设置的最大长度
    dtype=dtype,  # 使用前面设置的数据类型
    load_in_4bit=load_in_4bit,  # 使用4位量化
    # token="hf_...",  # 如果需要访问授权模型，可以在这里填入密钥
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
prompt_style = """以下是描述任务的指令，以及提供进一步上下文的输入。
请写出一个适当完成请求的回答。
在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。

### 指令：
你是一位精通卜卦、星象和运势预测的算命大师。
请回答以下算命问题。

### 问题：
{}

### 回答：
<think>{}"""
# 定义提示风格的字符串模板，用于格式化问题

question = "1995年8月26日生，男，名字叫石天成，想了解健康运势"
# 定义具体的算命问题
FastLanguageModel.for_inference(model)
# 准备模型以进行推理

inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")
# 使用 tokenizer 对格式化后的问题进行编码，并移动到 GPU

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
# 使用模型生成回答

response = tokenizer.batch_decode(outputs)
# 解码模型生成的输出为可读文本

print(response[0])
# 打印生成的回答部分

<｜begin▁of▁sentence｜>以下是描述任务的指令，以及提供进一步上下文的输入。
请写出一个适当完成请求的回答。
在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。

### 指令：
你是一位精通卜卦、星象和运势预测的算命大师。
请回答以下算命问题。

### 问题：
1995年8月26日生，男，名字叫石天成，想了解健康运势

### 回答：
<think>
嗯，石天成先生今年35岁，8月26日出生，想了解自己的健康运势。首先，我需要了解他的生日元素，也就是他的星座和运势。8月26日的人属于乙巳年，生于蛇年，属蛇。蛇在中国文化中象征着智慧、灵动和力量，同时也代表着医生和治疗的力量。

接下来，考虑他的健康运势。蛇的象征通常与健康和治疗相关，所以这可能意味着他在健康方面会有一定的机缘和天赋。他可能在医疗或健康领域有潜力，或者他对自己的身体管理有独特的理解。

考虑到他的星座特点，蛇的代表可能带来灵敏、敏锐的感知力和对细节的关注力，这对于健康管理非常重要。他可能对自己的身体状态有较高的敏感度，能够及时发现问题并采取措施。

此外，蛇的运势还可能赋予他一定的恢复能力和抗病力，这意味着在面对健康挑战时，他可能会表现出强大的内在力量和快速的恢复能力。

不过，我也需要注意到蛇的另一面，可能会有一定的警告或挑战。过度关注健康可能会导致压力或焦虑，或者他可能对某些健康问题过于敏感，导致过度担忧。因此，他需要在健康管理中找到平衡，避免过度消耗自己。

总的来说，石天成先生的健康运势看起来是积极的，他可能在健康方面有独特的机缘和潜力，能够利用自己的智慧和敏感来管理自己的身体和生活。但他也需要注意保持平衡，避免过度的压力和焦虑。
</think>

石天成先生今年35岁，8月26日出生，属于乙巳年、蛇年。蛇的代表通常与智慧、灵动和力量相关，同时也与医生和治疗相关，这可能意味着他在健康方面有独特的机缘和潜力。

根据他的星座特点，蛇象征着灵敏、敏锐的感知力和对细节的关注力，这对于健康管理非常重要。他可能对自己的身体状态有较高的敏感度，能够及时发现问题并采取措施。此外，蛇的运势还赋予他一定的恢复能力和抗病力，面对健康挑战时，他可能会表现出强大的内在力量和快速的恢复能力。

然而，蛇的另一面也需要注意，可能会有一定的警告或挑战。过度关注健康可能会导致压

## 加载数据集

In [ ]:
# 定义一个用于格式化提示的多行字符串模板
train_prompt_style = """以下是描述任务的指令，以及提供进一步上下文的输入。
请写出一个适当完成请求的回答。
在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。

### 指令：
你是一位精通八字算命、 紫微斗数、 风水、易经卦象、塔罗牌占卜、星象、面相手相和运势预测等方面的算命大师。
请回答以下算命问题。

### 问题：
{}

### 回答：
<思考>
{}
</思考>
{}"""

# 定义结束标记（EOS_TOKEN），用于指示文本的结束
EOS_TOKEN = tokenizer.eos_token  # 必须添加结束标记

# 导入数据集加载函数
from datasets import load_dataset
# 加载指定的数据集，选择中文语言和训练集的前500条记录
dataset = load_dataset("Conard/fortune-telling", 'default', split = "train[0:200]", trust_remote_code=True)
# 打印数据集的列名，查看数据集中有哪些字段
print('\n\n\ncolumn_names')
print(dataset.column_names)

# 定义一个函数，用于格式化数据集中的每条记录
def formatting_prompts_func(examples):
    # 从数据集中提取问题、复杂思考过程和回答
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []  # 用于存储格式化后的文本
    # 遍历每个问题、思考过程和回答，进行格式化
    for input, cot, output in zip(inputs, cots, outputs):
        # 使用字符串模板插入数据，并加上结束标记
        # 这是最容易被忽略但又极其重要的一步。 EOS 代表 End Of Sentence。
        # 如果不加这个标记，模型在训练时会认为答案永远没写完，导致它在以后预测时会变成一个“话痨”，不断重复废话或者陷入死循环。
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)  # 将格式化后的文本添加到列表中
    return {
        "text": texts,  # 返回包含所有格式化文本的字典
    }

dataset = dataset.map(formatting_prompts_func, batched = True)
print('\n\n\ndataset example')
dataset["text"][0]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Conard/fortune-telling' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Conard/fortune-telling' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.





column_names
['Question', 'Response', 'Complex_CoT']


Map:   0%|          | 0/200 [00:00<?, ? examples/s]




dataset example


'以下是描述任务的指令，以及提供进一步上下文的输入。\n请写出一个适当完成请求的回答。\n在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。\n\n### 指令：\n你是一位精通八字算命、 紫微斗数、 风水、易经卦象、塔罗牌占卜、星象、面相手相和运势预测等方面的算命大师。\n请回答以下算命问题。\n\n### 问题：\n新房装修,大门对着电梯好不好?要如何化解?\n\n### 回答：\n<思考>\n好的，用户问的是新房装修时大门对着电梯好不好，以及如何化解。首先，我需要回忆一下风水学中关于大门和电梯的相关知识。电梯在风水中属于动气比较强的地方，因为电梯频繁开合，会带来不稳定的气流，也就是所谓的“煞气”。大门是住宅的纳气口，如果正对电梯，可能会让这些不稳定的气流直接冲进家里，影响居住者的健康和财运。\n\n接下来，我需要确认用户的具体情况。比如，大门和电梯的距离有多远？是否正对还是稍微偏一点？不过用户没有提供这些细节，所以只能给出一般性的建议。化解的方法通常有几种：屏风或玄关、门帘、五帝钱、植物、八卦镜等。需要逐一解释这些方法的原理和使用方式，同时提醒用户要根据实际情况选择，必要时咨询专业风水师。\n\n另外，还要注意语气要亲切，避免使用过于专业的术语，让用户容易理解。同时，要强调这些是传统方法，效果因人而异，保持客观中立。最后，可以建议用户如果情况复杂，最好请专业人士实地查看，这样更稳妥。\n\n</思考>\n根据传统风水学的观点，大门正对电梯易形成"开口煞"，电梯频繁升降会扰乱家宅气场。建议化解方案：\n\n1. 玄关阻隔法\n在入门处设置L型屏风或文化砖玄关墙，高度以1.8米为宜，既保持采光又形成缓冲带\n\n2. 五行通关法\n门槛石下埋设五帝钱+白玉葫芦，建议选丙申年铸造的真品古币，配合门楣悬挂九宫八卦镜\n\n3. 光影化解术\n安装磨砂玻璃内推门，门框镶嵌黄铜门槛，每日辰时用海盐净化门廊区域\n\n4. 现代科技方案\n入户区安装智能感应灯带，设置循环播放的流水声效，运用声光电技术平衡磁场\n\n需注意电梯井方位与家主命卦的关系，建议提供具体户型平面图进行吉凶方位测算。当代建筑中可采用半透明艺术隔断结合空气净化系统，既符合科学原理又兼顾传统智慧。<｜end▁of▁sentence｜>'

## 模型微调

In [ ]:
FastLanguageModel.for_training(model)

model = FastLanguageModel.get_peft_model(
    model,  # 传入已经加载好的预训练模型
    r = 16,  # 设置 LoRA 的秩，决定添加的可训练参数数量
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",  # 指定模型中需要微调的关键模块
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,  # 设置 LoRA 的超参数，影响可训练参数的训练方式
    lora_dropout = 0,  # 设置防止过拟合的参数，这里设置为 0 表示不丢弃任何参数
    bias = "none",    # 设置是否添加偏置项，这里设置为 "none" 表示不添加
    use_gradient_checkpointing = "unsloth",  # 使用优化技术节省显存并支持更大的批量大小
    random_state = 3407,  # 设置随机种子，确保每次运行代码时模型的初始化方式相同
    use_rslora = False,  # 设置是否使用 Rank Stabilized LoRA 技术，这里设置为 False 表示不使用
    loftq_config = None,  # 设置是否使用 LoftQ 技术，这里设置为 None 表示不使用
)

from trl import SFTTrainer  # 导入 SFTTrainer，用于监督式微调
from transformers import TrainingArguments  # 导入 TrainingArguments，用于设置训练参数
from unsloth import is_bfloat16_supported  # 导入函数，检查是否支持 bfloat16 数据格式

# trainer = SFTTrainer(  # 创建一个 SFTTrainer 实例
#     model=model,  # 传入要微调的模型
#     tokenizer=tokenizer,  # 传入 tokenizer，用于处理文本数据
#     train_dataset=dataset,  # 传入训练数据集
#     dataset_text_field="text",  # 指定数据集中文本字段的名称
#     max_seq_length=max_seq_length,  # 设置最大序列长度
#     dataset_num_proc=2,  # 设置数据处理的并行进程数
#     packing=False,  # 是否启用打包功能（这里设置为 False，打包可以让训练更快，但可能影响效果）
#     args=TrainingArguments(  # 定义训练参数
#         per_device_train_batch_size=2,  # 每个设备（如 GPU）上的批量大小
#         gradient_accumulation_steps=4,  # 梯度累积步数，用于模拟大批次训练
#         warmup_steps=5,  # 预热步数，训练开始时学习率逐渐增加的步数
#         max_steps=75,  # 最大训练步数
#         learning_rate=2e-4,  # 学习率，模型学习新知识的速度
#         fp16=not is_bfloat16_supported(),  # 是否使用 fp16 格式加速训练（如果环境不支持 bfloat16）
#         bf16=is_bfloat16_supported(),  # 是否使用 bfloat16 格式加速训练（如果环境支持）
#         logging_steps=1,  # 每隔多少步记录一次训练日志
#         optim="adamw_8bit",  # 使用的优化器，用于调整模型参数
#         weight_decay=0.01,  # 权重衰减，防止模型过拟合
#         lr_scheduler_type="linear",  # 学习率调度器类型，控制学习率的变化方式
#         seed=3407,  # 随机种子，确保训练结果可复现
#         output_dir="outputs",  # 训练结果保存的目录
#         report_to="none",  # 是否将训练结果报告到外部工具（如 WandB），这里设置为不报告
#     ),
# )



Unsloth 2026.1.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
## 显存优化版本
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 1024, # 降半处理，预防 OOM
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        # --- 核心显存优化参数 ---
        per_device_train_batch_size = 1, # 降低单次消耗
        gradient_accumulation_steps = 8, # 补偿批大小 (1x8=8)

        # --- 训练节奏控制 ---
        warmup_steps = 5,
        max_steps = 75,
        learning_rate = 2e-4,

        # --- 精度与硬件适配 ---
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "adamw_8bit",

        # --- 监控与保存 ---
        logging_steps = 1,
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        output_dir = "outputs",
        report_to = "none",

        # --- 稳定性核心（补回） ---
        seed = 3407, # 保证训练可复现
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 3 | Total steps = 75
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,2.340700
2,2.245600
3,2.462100
4,2.191400
5,2.022100
6,2.213200
7,1.950500
8,1.964600
9,1.845500
10,1.859800


## 微调之后的模型再跑一次

In [ ]:
# 将模型切换到推理模式，准备回答问题
FastLanguageModel.for_inference(model)

# 将问题转换成模型能理解的格式，并发送到 GPU 上
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# 让模型根据问题生成回答，最多生成 4000 个新词
outputs = model.generate(
    input_ids=inputs.input_ids,  # 输入的数字序列
    attention_mask=inputs.attention_mask,  # 注意力遮罩，帮助模型理解哪些部分重要
    max_new_tokens=4000,  # 最多生成 4000 个新词
    use_cache=True,  # 使用缓存加速生成
)

# 将生成的回答从数字转换回文字
response = tokenizer.batch_decode(outputs)

# 打印回答
print(response[0])

<｜begin▁of▁sentence｜>以下是描述任务的指令，以及提供进一步上下文的输入。
请写出一个适当完成请求的回答。
在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。

### 指令：
你是一位精通卜卦、星象和运势预测的算命大师。
请回答以下算命问题。

### 问题：
1995年8月26日生，男，名字叫石天成，想了解健康运势

### 回答：
<think>

<思考>
好的，用户提供了一个具体的请求，需要根据提供的信息进行分析。首先，用户的生辰是1995年8月26日，性别是男性，名字叫石天成。用户的主要问题是想了解健康运势。根据用户的要求，需要结合传统命理（紫微斗数）来分析。

首先，确定用户的生辰八字。1995年是乙丑年，8月是庚寅月，26日是甲辰日。具体排盘需要查万年历，确认具体的年柱、月柱、日柱、时柱。但由于用户没有提供时辰，可能需要使用默认的时辰（比如12点）或者不计时辰。这里假设时辰为未时（9点-11点），对应丁未时。

接下来，分析生辰八字的五行分布。1995年乙丑年是土木年，8月庚寅月是土木月，26日甲辰日是金土日。时辰丁未时是土火时。整体来看，八字可能偏土木元素较多，金、水、火、木、土五行平衡，需要根据具体排盘调整。

然后，健康运势需要结合八字中的天干地支与五行分布。辰土为身宫，乙木为官星，寅木为月支，甲金为日干，丁火为时干。辰土为身宫，寅木为月干，甲金为日干，丁火为时干。八字中有金、火、木、土四行，水缺乏，可能需要注意水元素的调和。

根据传统命理学，健康运势需要看八字中的神煞、煞星、害神等。例如，辰土为身宫，寅木为月干，甲金为日干，丁火为时干。辰土为身宫，寅木为月干，甲金为日干，丁火为时干。需要结合这些星曜的作用来判断健康运势。

另外，还需要考虑流年和大运的影响。1995年是乙丑年，1996年是丙寅年，1997年是丁卯年，1998年是戊午年，1999年是己未年，2000年是庚申年。这些年份的天干地支与用户的生辰八字相互作用，可能会影响健康运势的变化。

在分析健康运势时，需要注意以下几个方面：
1. 五行平衡：土木较多，水缺乏，可能需要注意肾脏、泌尿系统等水相关的健康问题。
2. 煞星影响：辰土为身宫，寅木为月干，甲金为日干，丁火为时干，需要结合这些星曜的性质来分析健康风险。
3. 流年影响：不同的大

## 可能的报错
显存OOM

In [ ]:
## trainer_stats = trainer.train() 报错：OOM

解决方法：
1. 降低 max_seq_length（最有效）
将 max_seq_length 从 2048 降低到 512 或 1024。 算命的对话通常不会特别长，1024 足够覆盖大部分推演过程。
2. 启用极速优化项（Unsloth 专用）
```python
# 在 TrainingArguments 中修改/添加：
per_device_train_batch_size = 1,  # 从 2 降到 1，最直接减少显存压力
gradient_accumulation_steps = 8,  # 将累积步数翻倍（1x8=8），保持总 Batch Size 不变
```
3. restart session! 必须重启